# Re-Identificator Training - PersonViT on PRW

| ID         | Student         | Email                              |
|------------|-----------------|------------------------------------|
| 0001189110 | Rimondi Simone  | simone.rimondi5@studio.unibo.it    |


## 1. Introduction


Person re-identification (Re-ID) is the problem of determining whether two images of a person, acquired from different cameras or at different points in time, depict the same individual. Unlike conventional classification, where the set of classes is fixed and known at training time, Re-ID operates in an *open-set* regime: the identities present in the gallery at deployment have never been observed during training. The model must therefore learn not a classifier, but a *distance function* over an embedding space such that representations of the same identity cluster together while representations of distinct identities are pushed apart, regardless of confounding factors such as viewpoint, illumination, or partial occlusion.

This notebook addresses the problem in the context of a **two-stage person search pipeline** applied to the **PRW** (*Person Re-identification in the Wild*) benchmark. The first stage, developed in the detection notebook, produces pedestrian bounding boxes via YOLO26. The present notebook constitutes the **second stage**: each detected crop is projected into a high-dimensional metric space, and matching against a query is performed by ranking gallery embeddings according to cosine similarity. The experimental objective is to identify the optimal fine-tuning configuration for **PersonViT** on the PRW target domain, starting from checkpoints pre-trained on established Re-ID source datasets.


### Architecture: PersonViT and the TransReID Framework


PersonViT is a re-identification model built on the **TransReID** framework, which adapts a pure Vision Transformer (ViT) to the metric learning task through three key architectural modifications.

**1. ViT backbone with patch embedding.** The input image (256x128 pixels, the 2:1 aspect ratio typical of pedestrian crops) is divided into non-overlapping 16x16 patches, producing a sequence of tokens. A learnable `[CLS]` token is prepended to this sequence; its final hidden state serves as the global identity representation. The model is available in two scales:

| Variant    | Embedding dim | Heads | Layers | Approx. parameters |
|------------|--------------|-------|--------|---------------------|
| ViT-Base   | 768          | 12    | 12     | 86 M               |
| ViT-Small  | 384          | 6     | 12     | 22 M               |

**2. Side Information Embedding (SIE).** TransReID introduces a learnable additive embedding that encodes contextual metadata (camera ID, viewpoint index), summed to the patch embedding before the transformer stack. This mechanism allows the model to disentangle identity-driven variation from context-driven variation during training, improving cross-camera generalisation. In this notebook, SIE is configured with `camera_num=0` and `view_num=0` because PRW does not provide structured viewpoint annotations; the architecture is otherwise identical to the one used on the source datasets.

**3. BNNeck and final projection.** After the transformer, the `[CLS]` token passes through a Batch Normalization Neck (BNNeck) before the final embedding projection. The original classifier head, used during pre-training with cross-entropy supervision, is replaced by `nn.Identity()` to expose the raw embedding, which is then L2-normalised before cosine distance computation. This decoupling between feature learning and the metric loss is essential for the open-set deployment regime.

The Jigsaw Patch Module (JPM), an optional TransReID component that introduces spatial permutations of patches to encourage spatially localised features, is disabled throughout this study (`JPM=False`) in order to isolate the effect of the experimental variables of interest without introducing additional architectural variance.


### Strengths of this Notebook


Compared to a standard fine-tuning approach, several deliberate design choices increase the experimental value and reproducibility of this work:

Three families of metric learning loss (**TripletMarginLoss**, **ArcFaceLoss**,
**NTXentLoss**) are exposed through a unified interface via pytorch-metric-learning, ensuring that comparisons across losses are free of implementation-level confounders.
- **Orthogonal ablation axes.** The three experimental variables (source checkpoint, fine-tuning strategy, loss function) are varied one at a time, with the others fixed to the best value identified in the preceding phase. This sequential scheme reduces the number of required runs from O(N^3) to O(N) while preserving causal interpretability of each result.
- **Rigorous Re-ID evaluation protocol.** The `evaluate_reid` function correctly excludes the query from the gallery based on filename matching, not merely on PID, and computes the Cumulative Matching Characteristic (CMC) up to Rank-50 alongside mean Average Precision (mAP). The mAP metric penalises correct matches ranked too low in the retrieval list and provides a more complete picture of embedding quality than Rank-1 alone.
- **Incremental result persistence.** Every run is serialised to `all_results.json` immediately upon completion, making the notebook resilient to the disconnections typical of Kaggle sessions.
- **Efficient small-first ablation with scale-up validation.** The complete ablation over
  fine-tuning strategies and loss functions is run exclusively on ViT-Small (22 M
  parameters), keeping each experimental iteration fast and resource-light. Phase 3 then
  *scales up* the winning configuration to ViT-Base (86 M parameters), producing a
  cost-benefit analysis (mAP vs. GFLOPs vs. inference latency) that is directly actionable
  when selecting which model variant to deploy in the full person search pipeline.



### Dataset: PRW


**PRW** (*Person Re-identification in the Wild*) is a surveillance dataset collected by six static cameras installed on the campus of Tsinghua University. It comprises 11,816 frames, 932 labelled identities, and 34,304 bounding-box annotations for the *person* class. Unlike classic Re-ID benchmarks such as Market-1501, where crops are already pre-extracted, PRW provides full frames with MATLAB annotations, requiring a preprocessing step to extract pedestrian crops. Annotations include ambiguous identities (pid -2), which in this study are retained as negative samples during training to increase the discriminative pressure on the metric loss.

The official dataset split is as follows:
- **Train**: annotated frames with known identities, used for fine-tuning
- **Gallery (Test)**: frames against which queries are matched at evaluation time
- **Query**: pre-extracted crops of target identities to be retrieved from the gallery


### Methodology


The experimental pipeline is organised into four sequential phases, summarised in the table
below. The ablation study (Phases 0–2) is deliberately conducted on **ViT-Small** (22 M
parameters) rather than ViT-Base (86 M parameters). Because ViT-Small is approximately 4×
lighter, each training epoch completes in roughly one-quarter of the time, making it
practical to sweep three fine-tuning strategies and three loss functions within a single
Kaggle session. Only after the optimal (strategy, loss) pair has been identified does the
study commit to a single full-scale training run on ViT-Base (Phase 3). This *small-first,
scale-up* design reduces total GPU time by approximately 3× compared with running the
complete ablation on ViT-Base, while preserving full causal interpretability: because the
two architectures share the same TransReID backbone design and pre-training regime, the
configuration that maximises mAP on ViT-Small is a reliable proxy for the configuration
that will generalise to the larger model.

| Phase | Description |
|-------|-------------|
| 0     | Pretrained Baselines – Zero-shot evaluation of all four **ViT-Small** checkpoints on PRW |
| 1     | Strategy Comparison – Full FT vs Partial FT vs FreezeRetrain, ArcFace loss fixed (**ViT-Small**) |
| 2 | Loss Comparison – TripletMarginLoss vs ArcFaceLoss vs **NTXentLoss**, best strategy fixed (ViT-Small) |
| 3     | Scale-Up – Winning configuration replicated on **ViT-Base** (embedding dim 768) |
| Final | Cross-model comparison: metrics, params, GFLOPs, latency |



### Reproducibility and Runtime Environment


All experiments use a fixed global seed (42) applied to Python, NumPy, and PyTorch. The notebook runs on a single **NVIDIA T4 GPU** (16 GB VRAM) within a Kaggle environment, with fp16 mixed-precision training via `torch.cuda.amp`. Best checkpoints (by mAP) are saved automatically during training and reloaded for final evaluation. All metrics are exported as a CSV table at the end of the notebook for downstream analysis.


## 2. Preliminaries Operations

### Installation

In [ ]:
!pip install -q albumentations opencv-python-headless scipy
!pip install -q torchmetrics timm einops yacs
!pip install -q pytorch-metric-learning
!pip install -q thop

### Imports

In [ ]:
import os
import cv2
import copy
import json
import math
import time
import random
import sys
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import numpy as np
import scipy.io
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.gridspec as gridspec
from matplotlib.ticker import MultipleLocator
from matplotlib import rcParams
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from pytorch_metric_learning import losses as pml_losses
from pytorch_metric_learning import miners as pml_miners

from thop import profile as thop_profile, clever_format

print('All imports OK')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

### Import PersonViT Model

The model is cloned from a personal GitHub fork (`simoswish02/PersonViT`) rather than the original upstream source for two reasons: Kaggle sessions do not persist cloned directories across restarts, so a controlled fork guarantees version stability; and the original repository contained an import error in `transreid_pytorch/model/make_model.py` that prevented the module from loading correctly outside of its own directory.

The fix consists of replacing the original **implicit import**:

    from loss.metric_learning import Arcface, Cosface, AMSoftmax, CircleLoss

with a **fully-qualified absolute import**:

    from transreid_pytorch.loss.metric_learning import (
        Arcface, Cosface, AMSoftmax, CircleLoss
    )

The original line assumes `loss` is a top-level package on `sys.path`, which only holds when the interpreter is launched from inside the `transreid_pytorch` directory itself. Since this notebook inserts `PersonViT/` into `sys.path`, the correct top-level namespace becomes `transreid_pytorch`, and the corrected import resolves unambiguously under any working directory.


In [ ]:
if not os.path.exists('PersonViT'):
    os.system('git clone https://github.com/simoswish02/PersonViT')
    print('PersonViT cloned')
else:
    print('PersonViT already present')

sys.path.insert(0, 'PersonViT')
from transreid_pytorch.config import cfg as reid_cfg
from transreid_pytorch.model.make_model import make_model
print('TransReID imports OK')

### Centralised Configuration

All experimental hyperparameters are defined in a single `Config` class, which acts as the unique source of truth for the entire notebook. This design ensures that any change to a hyperparameter, whether a learning rate, a loss margin, or a file path, propagates automatically to every cell that references `cfg`, eliminating the risk of inconsistencies across phases. The block below also initialises all random seeds and creates the output directories required for checkpoints, plots, and result persistence.

In [ ]:
class Config:
    # Dataset
    dataset_root = '/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild'

    # Pretrained checkpoints ViT-Base
    pretrained_models = {
        'duke': {
            'path': '/kaggle/input/models/simonerimondi/personvit/pytorch/default/4/PersonViT_base_pretrained_duke.pth',
            'display_name': 'Duke',
        },
        'market1501': {
            'path': '/kaggle/input/models/simonerimondi/personvit/pytorch/default/4/PersonViT_base_pretrained_market.pth.pth',
            'display_name': 'Market-1501',
        },
        'msmt': {
            'path': '/kaggle/input/models/simonerimondi/personvit/pytorch/default/4/PersonViT_base_pretrained_msmt.pth',
            'display_name': 'MSMT17',
        },
        'occ_duke': {
            'path': '/kaggle/input/models/simonerimondi/personvit/pytorch/default/4/PersonViT_base_pretrained_occ_duke.pth',
            'display_name': 'Occluded-Duke',
        },
    }
    best_pretrain_key = 'market1501'

    # Pretrained checkpoints ViT-small
    pretrained_models_small = {
        'duke': {
            'path': '/kaggle/input/models/simonerimondi/personvit/pytorch/default/4/PersonViT_small_pretrain_duke.pth',
            'display_name': 'Duke',
        },
        'market1501': {
            'path': '/kaggle/input/models/simonerimondi/personvit/pytorch/default/4/PersonViT_small_pretrained_market.pth',
            'display_name': 'Market-1501',
        },
        'msmt': {
            'path': '/kaggle/input/models/simonerimondi/personvit/pytorch/default/4/PersonViT_small_pretrained_msmt.pth',
            'display_name': 'MSMT17',
        },
        'occ_duke': {
            'path': '/kaggle/input/models/simonerimondi/personvit/pytorch/default/4/PersonViT_small_pretrained_occ_duke.pth',
            'display_name': 'Occluded-Duke',
        },
    }

    # ViT-Base settings
    vit_base_transformer_type = 'vit_base_patch16_224_TransReID'
    vit_base_embedding_dim = 768

    # ViT-Small settings
    vit_small_transformer_type = 'vit_small_patch16_224_TransReID'
    vit_small_embedding_dim = 384

    # Augmentation
    img_height = 256
    img_width = 128
    aggressive_augmentation = True

    # DataLoader
    batch_size = 128
    num_workers = 4

    # Evaluation
    exclude_query_from_gallery = True
    use_ambiguous_as_negatives = True

    # Metric learning losses
    triplet_margin = 0.3
    arcface_margin = 30
    arcface_scale = 64.0
    angular_alpha = 40

    # Mixed precision
    use_amp = True

    # Fine-tuning LR / schedule
    full_lr=1e-5;    full_wd=1e-4;    full_epochs=20;    full_warmup_ep=3;    full_warmup_factor=0.01
    partial_lr=3e-4; partial_wd=1e-4; partial_epochs=20; partial_warmup_ep=3; partial_warmup_factor=0.1
    freeze_lr=3e-4;  freeze_wd=1e-4;  freeze_epochs=20;  freeze_warmup_ep=3;  freeze_warmup_factor=0.1

    # Reproducibility and output paths
    seed        = 42
    save_dir    = '/kaggle/working/personvit_finetuning'
    results_dir = Path('./evaluation_results')
    plots_dir   = results_dir / 'plots'

# Instantiate config and create output directories
cfg = Config()
os.makedirs(cfg.save_dir, exist_ok=True)
cfg.results_dir.mkdir(exist_ok=True)
cfg.plots_dir.mkdir(exist_ok=True)

# Fix all random seeds for reproducibility
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)

# Initialise the results registry (populated incrementally during training)
RESULTS = {}
RESULTS_PATH = cfg.results_dir / 'all_results.json'

print('='*70)
print('CONFIGURATION')
print('='*70)
print(f'  Device       : {device}')
print(f'  AMP          : {cfg.use_amp}')
print(f'  Batch size   : {cfg.batch_size}')
print(f'  ViT-Base emb : {cfg.vit_base_embedding_dim}')
print(f'  ViT-Small emb: {cfg.vit_small_embedding_dim}')
print('='*70)

### Results Registry and Persistence

The three functions below manage the serialisation and deserialisation of the global `RESULTS` dictionary, which accumulates evaluation metrics and profiling data for every run throughout the notebook.

- `_json_serialisable(v)`: a recursive helper that converts NumPy scalar types (`np.floating`, `np.integer`) to native Python types before serialisation, since the standard `json` module does not support NumPy dtypes natively. Dictionaries and lists are traversed recursively; all other types are returned unchanged.
- `save_results()`: serialises the current state of `RESULTS` to a formatted JSON file at `RESULTS_PATH`, stripping the `history` key from each run to avoid persisting large per-batch loss arrays. It is called immediately after every training run so that progress is preserved even in the event of a Kaggle session timeout.
- `load_results()`: reloads `RESULTS` from the JSON file on disk, restoring the full registry without re-running any experiment. It is primarily intended for use after a kernel restart, provided the JSON file is still available in the working directory.

In [ ]:
def _json_serialisable(v):
    if isinstance(v, (float, np.floating)): return round(float(v), 6)
    if isinstance(v, (int,   np.integer)):  return int(v)
    if isinstance(v, dict):  return {kk: _json_serialisable(vv) for kk, vv in v.items()}
    if isinstance(v, list):  return [_json_serialisable(x) for x in v]
    return v

def save_results():
    serialisable = {
        k: _json_serialisable({kk: vv for kk, vv in v.items() if kk != 'history'})
        for k, v in RESULTS.items()
    }
    with open(RESULTS_PATH, 'w') as f:
        json.dump(serialisable, f, indent=2)
    print(f'[checkpoint] RESULTS saved to {RESULTS_PATH}  ({len(RESULTS)} runs)')

def load_results():
    if RESULTS_PATH.exists():
        with open(RESULTS_PATH) as f:
            RESULTS.update(json.load(f))
        print(f'Loaded {len(RESULTS)} runs from {RESULTS_PATH}')
    else:
        print('No previous results found')

## 3. Dataset Utilities

### Dataset Classes

Three `Dataset` subclasses handle the different access patterns required by the PRW pipeline. `PRWPersonSearchDataset` is the base loader and operates in three modes (`train`, `test`, `query`), parsing the MATLAB annotation files provided by PRW and converting each bounding box to a standardised dictionary record. `PRWReIDDataset` and `QueryDataset` wrap it with transform support for the actual training and evaluation loops.

- `PRWPersonSearchDataset`: loads the raw PRW dataset from disk. In `train`/`test` mode it reads the official split `.mat` file, iterates over per-frame annotation files, and filters out boxes smaller than `min_box_area` pixels to discard degenerate annotations. In `query` mode it reads `query_info.txt` and loads pre-extracted query crops directly. Each annotation record stores the image path, bounding boxes in `[x1, y1, x2, y2]` format, person IDs, camera ID, and frame dimensions.
- `_build_annotations()`: core preprocessing method called during `train`/`test` initialisation. For each frame it reads the `box_new` field from the MATLAB annotation, clips coordinates to image boundaries, and discards boxes that do not meet the minimum area threshold. Ambiguous identities (pid `-2`) are retained and counted separately.
- `PRWReIDDataset`: wraps `PRWPersonSearchDataset` to expose individual crops as training samples. It flattens all frame-level annotations into a flat list of `(frame_index, box, pid, filename)` tuples and builds a contiguous integer label mapping (`pid_to_label`) required by the metric learning losses. Ambiguous identities can optionally be included as additional negatives via `include_ambiguous`.
- `QueryDataset`: a thin wrapper around `PRWPersonSearchDataset` in query mode that applies the test-time transform to each crop before returning it, making it compatible with the standard `DataLoader` interface used during evaluation.

In [ ]:
class PRWPersonSearchDataset(Dataset):
    def __init__(self, root, mode='train', min_box_area=100):
        self.root = root
        self.mode = mode.lower()
        self.min_box_area = min_box_area
        self.frames_dir = os.path.join(root, 'frames')
        self.ann_dir    = os.path.join(root, 'annotations')
        if mode == 'query':
            self.query_dir = os.path.join(root, 'query_box')
            self.query_info_file = os.path.join(root, 'query_info.txt')
            self._load_query_data()
        else:
            split_file = os.path.join(root, f'frame_{self.mode}.mat')
            if not os.path.exists(split_file):
                raise FileNotFoundError(f'Split file not found: {split_file}')
            self.ann_files  = sorted(self._load_filenames(split_file))
            self.annotations = []
            self._build_annotations()
        print(f'[{self.mode.upper()}] Loaded {len(self)} samples')

    def _load_filenames(self, path):
        mat = scipy.io.loadmat(path)
        key = [k for k in mat.keys() if not k.startswith('__')][0]
        out = []
        for item in mat[key].flatten():
            name = str(item[0]) if isinstance(item, np.ndarray) else str(item)
            ann  = name + '.jpg.mat'
            if os.path.exists(os.path.join(self.ann_dir, ann)):
                out.append(ann)
        return out

    def _extract_camera_id(self, img_name):
        try:
            if img_name.startswith('c'): return int(img_name[1])
        except Exception:
            pass
        return 1

    def _build_annotations(self):
        total = ambig = 0
        for af in tqdm(self.ann_files, desc=f'Loading {self.mode}'):
            img_name = af.replace('.mat', '').replace('.jpg', '') + '.jpg'
            img_path = os.path.join(self.frames_dir, img_name)
            mat_path = os.path.join(self.ann_dir, af)
            if not os.path.exists(img_path):
                continue
            img = cv2.imread(img_path)
            if img is None:
                continue
            H, W = img.shape[:2]
            vb, vp = [], []
            for box in scipy.io.loadmat(mat_path).get('box_new', []):
                pid = int(box[0])
                x1 = max(0, float(box[1]))
                y1 = max(0, float(box[2]))
                w = float(box[3])
                h  = float(box[4])
                x2  = min(W, x1+w)
                y2 = min(H, y1+h)
                if w > 1 and h > 1 and (x2-x1)*(y2-y1) >= self.min_box_area:
                    vb.append([x1,y1,x2,y2]); vp.append(pid)
                    total += 1
                    if pid == -2:
                        ambig += 1
            self.annotations.append({
                'img_name': img_name, 'img_path': img_path,
                'boxes': np.array(vb, dtype=np.float32) if vb else np.zeros((0,4), dtype=np.float32),
                'pids':  np.array(vp, dtype=np.int32)   if vp else np.zeros((0,),  dtype=np.int32),
                'cam_id': self._extract_camera_id(img_name), 'height': H, 'width': W,
            })
        print(f'  Boxes: {total}  Ambiguous: {ambig}')

    def _load_query_data(self):
        self.annotations = []
        with open(self.query_info_file) as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 6: continue
            pid = int(parts[0])
            if pid == -2:
                continue
            x,y,w,h = float(parts[1]),float(parts[2]),float(parts[3]),float(parts[4])
            orig_name = parts[5]+'.jpg'
            crop_name = f'{pid}_{orig_name}'
            crop_path = os.path.join(self.query_dir, crop_name)
            if not os.path.exists(crop_path):
                continue
            img = cv2.imread(crop_path)
            if img is None:
                continue
            ch, cw = img.shape[:2]
            self.annotations.append({
                'img_name': crop_name, 'img_path': crop_path,
                'boxes': np.array([[x,y,x+w,y+h]], dtype=np.float32),
                'pids':  np.array([pid], dtype=np.int32),
                'cam_id': self._extract_camera_id(orig_name), 'height': ch, 'width': cw,
            })

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img = cv2.imread(ann['img_path'])
        if img is None:
            raise ValueError(f'Image not found: {ann["img_path"]}')
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), ann

    def get_annotation(self, idx):
        return self.annotations[idx]


class PRWReIDDataset(Dataset):
    def __init__(self, base_dataset, transform=None, include_ambiguous=False):
        self.base_dataset = base_dataset
        self.transform = transform
        self.samples = []
        self.pid_to_samples = defaultdict(list)
        for ii in tqdm(range(len(base_dataset)), desc='Building Re-ID dataset'):
            ann = base_dataset.get_annotation(ii)
            for box, pid in zip(ann['boxes'], ann['pids']):
                if pid > 0 or (pid == -2 and include_ambiguous):
                    self.pid_to_samples[pid].append(len(self.samples))
                    self.samples.append((ii, box, pid, ann['img_name']))
        self.valid_pids = sorted(p for p in self.pid_to_samples if p > 0)
        self.pid_to_label = {pid: i for i, pid in enumerate(self.valid_pids)}
        self.num_classes = len(self.valid_pids)
        print(f'Re-ID: {len(self.samples)} crops | {self.num_classes} IDs')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ii, box, pid, img_name = self.samples[idx]
        ann = self.base_dataset.get_annotation(ii)
        img = cv2.imread(ann['img_path'])
        if img is None:
            raise ValueError(f'Not found: {ann["img_path"]}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        x1,y1 = max(0,int(box[0])), max(0,int(box[1]))
        x2,y2 = min(img.shape[1],int(box[2])), min(img.shape[0],int(box[3]))
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            crop = np.zeros((64,32,3), dtype=np.uint8)
        if self.transform:
            crop = self.transform(image=crop)['image']
        else:
            crop = torch.from_numpy(crop).permute(2,0,1).float()/255.
        return crop, self.pid_to_label.get(pid,-1), pid, img_name


class QueryDataset(Dataset):
    def __init__(self, base, transform):
        self.base=base; self.transform=transform
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        img, ann = self.base[idx]
        if self.transform: img = self.transform(image=img)['image']
        return img, ann

### Data Augmentation

`build_transforms()` returns a pair of Albumentations pipelines: one for training and one for evaluation. The test transform applies only resize and ImageNet normalisation, ensuring deterministic and reproducible evaluation. The training transform is controlled by `cfg.aggressive_augmentation` and is available in two variants.

- **Standard**: resize, horizontal flip, and mild colour jitter. Suitable as a lightweight baseline.
- **Aggressive**: a richer pipeline designed to simulate the photometric and geometric degradations typical of outdoor surveillance footage. It includes:
  - `ColorJitter`, `RandomBrightnessContrast`, and `HueSaturationValue` to cover a wide range of illumination conditions across cameras;
  - `CoarseDropout` (1 to 5 rectangular holes) as a proxy for random occlusion, encouraging the model to learn features that are not localised to a single body region;
  - `GaussNoise` / `ISONoise` and `MotionBlur` / `GaussianBlur` (applied via `OneOf` to avoid redundant degradations) to simulate sensor noise and motion artefacts;
  - `RandomShadow` and `RandomFog` to cover outdoor lighting edge cases such as cast shadows and low-visibility conditions.

All transforms normalise using ImageNet statistics (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`), consistent with the pre-training regime of the ViT backbone.


In [ ]:
def build_transforms(cfg):
    norm = A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    if cfg.aggressive_augmentation:
        train_tf = A.Compose([
            A.Resize(cfg.img_height, cfg.img_width),
            A.HorizontalFlip(p=0.5),
            A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15, p=0.6),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            A.HueSaturationValue(15, 30, 20, p=0.5),
            A.CoarseDropout(num_holes_range=(1,5), hole_height_range=(8,32),
                            hole_width_range=(8,16), p=0.5),
            A.OneOf([A.GaussNoise(std_range=(0.04,0.2),p=1),
                     A.ISONoise(color_shift=(0.01,0.05),intensity=(0.1,0.5),p=1)], p=0.4),
            A.OneOf([A.MotionBlur(blur_limit=5,p=1),
                     A.GaussianBlur(blur_limit=(3,7),p=1)], p=0.3),
            A.RandomShadow(shadow_roi=(0,0,1,1), num_shadows_limit=(1,2), p=0.3),
            A.RandomFog(fog_coef_range=(0.1,0.3), alpha_coef=0.1, p=0.2),
            norm, ToTensorV2(),
        ])
        print('AGGRESSIVE augmentation')
    else:
        train_tf = A.Compose([
            A.Resize(cfg.img_height, cfg.img_width), A.HorizontalFlip(p=0.5),
            A.ColorJitter(0.2,0.2,0.2,0.1,p=0.5), norm, ToTensorV2(),
        ])
        print('STANDARD augmentation')
    test_tf = A.Compose([A.Resize(cfg.img_height,cfg.img_width), norm, ToTensorV2()])
    return train_tf, test_tf

train_transform, test_transform = build_transforms(cfg)

### Dataset Instantiation and DataLoaders

The three dataset splits are instantiated here following the official PRW protocol. Each base `PRWPersonSearchDataset` is then wrapped by the appropriate crop-level dataset class before being passed to a `DataLoader`.

- The **train** loader uses `shuffle=True` and `drop_last=True` to ensure that every batch has a consistent size, which is required by batch-level metric learning losses such as ArcFace and TripletMargin. Ambiguous identities (pid `-2`) are included as additional negatives if `cfg.use_ambiguous_as_negatives` is set.
- The **test** and **query** loaders use `shuffle=False` to guarantee that feature extraction order is deterministic and consistent with the index arrays used in `evaluate_reid()`. Ambiguous identities are always included in the gallery to match the standard PRW evaluation protocol.
- `pin_memory=True` is set on all loaders to accelerate host-to-device transfers on the T4 GPU.


In [ ]:
train_base    = PRWPersonSearchDataset(cfg.dataset_root, mode='train')
test_base     = PRWPersonSearchDataset(cfg.dataset_root, mode='test')
query_base    = PRWPersonSearchDataset(cfg.dataset_root, mode='query')

train_reid    = PRWReIDDataset(train_base, transform=train_transform,
                               include_ambiguous=cfg.use_ambiguous_as_negatives)
test_reid     = PRWReIDDataset(test_base,  transform=test_transform, include_ambiguous=True)
query_dataset = QueryDataset(query_base, test_transform)

train_loader  = DataLoader(train_reid,    batch_size=cfg.batch_size,   shuffle=True,
                           num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
test_loader   = DataLoader(test_reid,     batch_size=cfg.batch_size,   shuffle=False,
                           num_workers=cfg.num_workers, pin_memory=True)
query_loader  = DataLoader(query_dataset, batch_size=cfg.batch_size,   shuffle=False,
                           num_workers=cfg.num_workers, pin_memory=True)

print(f'Train : {len(train_reid):,} crops | {train_reid.num_classes} IDs | {len(train_loader)} batches')
print(f'Test  : {len(test_reid):,} crops')
print(f'Query : {len(query_dataset):,} crops')

## 4. Training, Evaluation and Visualization Pipeline

### Model and Training Utilities

This section defines the factory functions used to instantiate, configure, and optimise PersonViT across all experimental phases. All training runs are assembled exclusively from these primitives, ensuring a consistent and reproducible setup.

- `create_personvit_model()`: instantiates a PersonViT model from a given checkpoint by deep-copying the TransReID config object and overriding only the fields relevant to the current run (architecture type, embedding dimension, input resolution, pre-trained weights path). The Jigsaw Patch Module is disabled (`JPM=False`) and `NECK_FEAT='after'` ensures the post-BNNeck embedding is used at inference time. The classification head is replaced with `nn.Identity()` to expose the raw embedding vector for metric learning.

- `freeze_all_backbone()`, `unfreeze_all()`, `freeze_backbone_unfreeze_head()`: the three fine-tuning strategy primitives used in Phase 1. They act on `model.base` (the ViT transformer stack) by toggling `requires_grad` on its parameters. `freeze_backbone_unfreeze_head()` combines both: it freezes the backbone and then re-enables gradients for all parameters outside `model.base` (BNNeck, projection layer).

- `param_stats()`: prints the ratio of trainable to total parameters for a given model configuration, providing a quick sanity check that the intended freeze strategy has been applied correctly.

- `build_optimizer_and_scheduler()`: constructs an AdamW optimiser over all trainable parameters, including any extra modules (e.g. the ArcFace weight matrix). The learning rate schedule combines a linear warm-up phase with cosine annealing, implemented as a step-level `LambdaLR` to allow sub-epoch scheduling granularity.

- `build_loss_and_miner` returns the loss function, an optional hard-pair miner, and an optional auxiliary optimiser for the three supported metric learning objectives:
  - **TripletMarginLoss** with `BatchHardMiner`, which selects the hardest positive and hardest negative in each batch;
  - **ArcFaceLoss**, a proxy-based loss that maintains a learnable class weight matrix and therefore requires its own AdamW optimiser updated jointly with the backbone;
  - **NTXentLoss** (Normalized Temperature-scaled Cross Entropy), a contrastive loss derived from SimCLR that maximises cosine similarity between positive pairs (same identity) while treating all remaining samples in the batch as negatives, normalised via a softmax scaled by temperature `cfg.ntxent_temperature`. Unlike the triplet losses, no explicit miner is required: all in-batch negatives are used implicitly, making the loss well-suited to large batches where many hard negatives are naturally present.



In [ ]:
def create_personvit_model(pretrain_path, transformer_type, embedding_dim,
                           num_classes, reid_cfg_obj):
    mc = copy.deepcopy(reid_cfg_obj)
    mc.MODEL.NAME             = 'transformer'
    mc.MODEL.TRANSFORMER_TYPE = transformer_type
    mc.MODEL.JPM              = False
    mc.MODEL.PRETRAIN_CHOICE  = 'self'
    mc.MODEL.PRETRAIN_PATH    = pretrain_path
    mc.MODEL.FEAT_DIM         = embedding_dim
    mc.INPUT.SIZE_TRAIN       = [cfg.img_height, cfg.img_width]
    mc.INPUT.SIZE_TEST        = [cfg.img_height, cfg.img_width]
    mc.TEST.NECK_FEAT         = 'after'
    model = make_model(mc, num_class=num_classes, camera_num=0, view_num=0)
    model.classifier = nn.Identity()
    return model.to(device)


def freeze_all_backbone(model):
    for p in model.base.parameters(): p.requires_grad = False

def unfreeze_all(model):
    for p in model.parameters(): p.requires_grad = True

def freeze_backbone_unfreeze_head(model):
    freeze_all_backbone(model)
    for name, p in model.named_parameters():
        if 'base.' not in name: p.requires_grad = True

def param_stats(model, tag=''):
    total    = sum(p.numel() for p in model.parameters())
    trainable= sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'[{tag}] trainable {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
    return total, trainable


def build_optimizer_and_scheduler(model, extra_modules, lr, wd,
                                  num_epochs, warmup_epochs, warmup_factor):
    params = [{'params': [p for p in model.parameters() if p.requires_grad], 'lr': lr}]
    for m in extra_modules:
        params.append({'params': list(m.parameters()), 'lr': lr})
    optimizer    = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    total_steps  = len(train_loader) * num_epochs
    warmup_steps = len(train_loader) * warmup_epochs

    def lr_lambda(step):
        if step < warmup_steps:
            return warmup_factor + (1.-warmup_factor)*step/max(warmup_steps,1)
        prog = (step-warmup_steps)/max(total_steps-warmup_steps,1)
        return 0.5*(1.+math.cos(math.pi*prog))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    print(f'AdamW  LR={lr}  WD={wd}  warmup={warmup_epochs} ep  total={num_epochs} ep')
    return optimizer, scheduler


def build_loss_and_miner(loss_type, num_classes, embedding_dim):
    if loss_type == 'triplet':
        return pml_losses.TripletMarginLoss(margin=cfg.triplet_margin), pml_miners.BatchHardMiner(), None
    if loss_type == 'arcface':
        loss_fn = pml_losses.ArcFaceLoss(
            num_classes=num_classes, embedding_size=embedding_dim,
            margin=cfg.arcface_margin, scale=cfg.arcface_scale,
        ).to(device)
        loss_opt = torch.optim.AdamW(loss_fn.parameters(), lr=1e-3, weight_decay=1e-4)
        return loss_fn, None, loss_opt
    if loss_type == 'angular':
        return pml_losses.AngularLoss(alpha=cfg.angular_alpha), pml_miners.AngularMiner(angle=cfg.angular_alpha), None
    raise ValueError(f'Unknown loss: {loss_type}')

### Training Loop and Fine-Tuning Orchestration

This section implements the full training pipeline through three layered functions, from the single-batch update up to the complete multi-phase experiment runner.

- `train_one_epoch()`: executes a single pass over the training set. Samples with label `-1` (identity not in the Re-ID label map) are masked out before the forward pass to avoid corrupting the metric loss. Embeddings are extracted from the first element of the model output tuple when the model returns multiple tensors. The miner, when present, selects informative pairs or triplets before passing indices to the loss function. The forward pass and loss computation run under `torch.amp.autocast`, which selectively casts operations to 16-bit floating point (fp16). Compared to the standard 32-bit representation (fp32), fp16 halves memory usage and exploits the T4 Tensor Cores for faster matrix multiplications, at the cost of a significantly reduced dynamic range: fp16 can only represent values down to approximately 6e-5, whereas fp32 reaches 1.2e-38. Gradients in deep networks are often very small and fall below this threshold, producing zeros (underflow) that silently corrupt the update. `GradScaler` addresses this by multiplying the loss by a large scalar factor before backpropagation, shifting gradient magnitudes into the representable fp16 range, and then dividing the gradients back down before the optimiser step. The same scaling is applied to the optional auxiliary optimiser required by ArcFaceLoss, which maintains its own learnable weight matrix. The scheduler is stepped at batch level to allow the cosine annealing schedule to operate at sub-epoch granularity. Per-batch losses and learning rates are recorded and returned for plotting.

- `train_model()`: orchestrates training over all epochs. A periodic evaluation is run every `eval_every` epochs and always at the final epoch; the model is explicitly set back to `train()` mode after each evaluation to avoid inadvertently leaving it in eval mode. The best checkpoint by mAP is saved to disk together with the ArcFace weight matrix state when applicable. All epoch-level and batch-level metrics are accumulated in the `history` dictionary, which is returned at the end and stored in `RESULTS` for later plotting.

- `run_finetuning()`: the top-level experiment runner that assembles a complete run from a strategy key and a loss type. It resolves all defaults (transformer type, embedding dimension, pretrained weights) so that the same function can be used for ViT-Base runs in Phases 1 and 2 and for the ViT-Small run in Phase 3 via `run_key_override`. If the target `run_key` is already present in `RESULTS`, the run is skipped entirely to avoid redundant computation when resuming a session. When `reset_head=True` (Freeze+Retrain strategy), all `Linear` and `BatchNorm1d` layers outside the backbone are re-initialised before training, ensuring that the head starts from a random state rather than inheriting source-domain weights. After training, the best checkpoint is reloaded before final evaluation to guarantee that reported metrics correspond to the best model seen during training and not the last epoch.



In [ ]:
def train_one_epoch(model, loss_fn, miner, optimizer, scheduler,
                   scaler, epoch, num_epochs, loss_optimizer=None):
    model.train()
    epoch_loss, batch_losses, batch_lrs = 0., [], []

    for images, labels, _pids, _fnames in tqdm(
            train_loader, desc=f'Ep {epoch+1}/{num_epochs}'):
        mask = labels >= 0
        if mask.sum() == 0: continue
        images, labels = images[mask].to(device), labels[mask].to(device)

        with torch.amp.autocast(device_type='cuda', enabled=cfg.use_amp):
            out        = model(images)
            embeddings = out[0] if isinstance(out, tuple) else out
            indices    = miner(embeddings, labels) if miner is not None else None
            loss       = (loss_fn(embeddings, labels, indices)
                         if indices is not None else loss_fn(embeddings, labels))

        optimizer.zero_grad()
        if loss_optimizer: loss_optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        if loss_optimizer: scaler.step(loss_optimizer)
        scaler.update()
        scheduler.step()

        batch_losses.append(loss.item())
        batch_lrs.append(optimizer.param_groups[0]['lr'])
        epoch_loss += loss.item()

    avg = epoch_loss / max(len(batch_losses), 1)
    return {'epoch_loss': avg, 'batch_losses': batch_losses, 'batch_lrs': batch_lrs}


def train_model(model, loss_fn, miner, optimizer, scheduler, num_epochs,
                loss_optimizer=None, checkpoint_name='best.pth', eval_every=5):
    print('\n' + '='*60 + f'\nFINE-TUNING: {checkpoint_name}\n' + '='*60)
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.use_amp)
    best_map = 0.
    history  = {'train_loss':[], 'lr':[], 'batch_loss':[], 'batch_lr':[],
                'eval_map':[], 'eval_rank1':[], 'eval_epochs':[]}

    for epoch in range(num_epochs):
        stats = train_one_epoch(model, loss_fn, miner, optimizer, scheduler,
                                scaler, epoch, num_epochs, loss_optimizer)
        history['train_loss'].append(stats['epoch_loss'])
        history['lr'].append(optimizer.param_groups[0]['lr'])
        history['batch_loss'].extend(stats['batch_losses'])
        history['batch_lr'].extend(stats['batch_lrs'])
        print(f'  Ep {epoch+1:>3}/{num_epochs}  '
              f'loss={stats["epoch_loss"]:.4f}  '
              f'lr={optimizer.param_groups[0]["lr"]:.2e}')

        if (epoch+1) % eval_every == 0 or epoch == num_epochs-1:
            res = run_evaluation(model, query_loader, test_loader)
            history['eval_map'].append(res['mAP'])
            history['eval_rank1'].append(res['rank1'])
            history['eval_epochs'].append(epoch+1)
            if res['mAP'] > best_map:
                best_map = res['mAP']
                ckpt = {'epoch': epoch+1, 'model_state_dict': model.state_dict(), 'results': res}
                if loss_optimizer: ckpt['loss_state_dict'] = loss_fn.state_dict()
                torch.save(ckpt, os.path.join(cfg.save_dir, checkpoint_name))
                print(f'  -> Best checkpoint saved (mAP {best_map*100:.2f}%)')
            model.train()

    print(f'\nDone. Best mAP: {best_map*100:.2f}%')
    return history

def run_finetuning(strategy_key, loss_type, transformer_type=None, embedding_dim=None, pretrain_path=None, run_key_override=None):
    sc = STRATEGY_CFG[strategy_key]
    t_type = transformer_type or cfg.vit_small_transformer_type
    emb_dim = embedding_dim or cfg.vit_small_embedding_dim
    pretrain = pretrain_path or cfg.pretrained_models_small[cfg.best_pretrain_key]['path']
    run_key = run_key_override or f'{strategy_key}_{loss_type}'
    print(t_type, emb_dim, pretrain, run_key)

    if run_key in RESULTS:
        print(f'[SKIP] {run_key} already in RESULTS - delete it to re-run.')
        return

    print(f'\n' + '#'*70)
    print(f'# RUN: {run_key}  |  {sc["label"]} x {loss_type.upper()}')
    print('#'*70)

    model = create_personvit_model(pretrain, t_type, emb_dim, train_reid.num_classes, reid_cfg)
    sc['freeze_fn'](model)

    if sc['reset_head']:
        for name, module in model.named_modules():
            if 'base.' not in name and isinstance(module, (nn.Linear, nn.BatchNorm1d)):
                module.reset_parameters()

    total_p, trainable_p = param_stats(model, run_key)

    loss_fn, miner, loss_opt = build_loss_and_miner(loss_type, train_reid.num_classes, emb_dim)
    extra = [loss_fn] if loss_opt is not None else []
    optimizer, scheduler = build_optimizer_and_scheduler(
        model, extra,
        lr=sc['lr'], wd=sc['wd'], num_epochs=sc['epochs'],
        warmup_epochs=sc['warmup_ep'], warmup_factor=sc['warmup_factor'])

    history = train_model(model, loss_fn, miner, optimizer, scheduler, sc['epochs'], loss_optimizer=loss_opt, checkpoint_name=f'best_{run_key}.pth', eval_every=5)

    # Reload best checkpoint
    ckpt_path = os.path.join(cfg.save_dir, f'best_{run_key}.pth')
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])

    final_res = run_evaluation(model, query_loader, test_loader)
    hw = profile_model(model)

    RESULTS[run_key] = {
        **final_res, **hw,
        'history':          history,
        'strategy':         sc['label'],
        'loss':             loss_type,
        'total_params':     total_p,
        'trainable_params': trainable_p,
    }
    save_results()

    plot_training_history(history, len(train_loader), sc['epochs'], save_path=os.path.join(cfg.save_dir, f'curves_{run_key}.png'), title_prefix = f'{sc["label"]} | {loss_type.capitalize()} - ',)
    print(f'\n{run_key} -> mAP {final_res["mAP"]*100:.2f}%  '
          f'Rank-1 {final_res["rank1"]*100:.2f}%  '
          f'Trainable params: {trainable_p:,}')

    del model; torch.cuda.empty_cache()

### Evaluation and Profiling

This section implements the full Re-ID evaluation pipeline and the efficiency profiling routine used across all experimental phases.

- `extract_features()`: runs inference over an entire dataloader in a single pass under `@torch.no_grad()` to suppress gradient computation and reduce memory usage. It handles both the four-element tuples returned by `PRWReIDDataset` and the two-element tuples returned by `QueryDataset`, making it compatible with both gallery and query loaders. Embeddings are L2-normalised before being accumulated, ensuring that cosine similarity can be computed as a simple dot product in `run_evaluation()`.

- `evaluate_reid()`: implements the standard CMC and mAP evaluation protocol for Re-ID. For each query, the gallery is ranked by ascending cosine distance. If `exclude_self=True`, gallery entries that originate from the same image as the query are removed from the ranked list before scoring; the check is performed on both the exact filename and the frame name embedded in the query crop filename (format `pid_framename.jpg`), correctly handling the PRW naming convention. The CMC curve is computed as the cumulative sum of a binary relevance vector capped at 1, averaged over all valid queries. The AP for each query is computed as the precision-at-k averaged only over positions where a correct match is retrieved, following the standard formula. Queries with no matching identity in the gallery are skipped and not counted towards `num_valid_queries`.

- `run_evaluation()`: orchestrates the full evaluation by calling `extract_features()` on both the query and gallery loaders, assembling the distance matrix as `1 - Q * G^T` (cosine distance from L2-normalised embeddings), and delegating scoring to `evaluate_reid()`.

- `profile_model()`: measures computational cost along two dimensions. Static complexity is estimated via `thop`, which counts multiply-accumulate operations (MACs) on a single dummy input; GFLOPs are reported as `2 * MACs` following the standard convention. Runtime latency is measured over `n_runs=100` forward passes after `n_warmup=10` warm-up iterations to allow the GPU to reach a stable clock state. On CUDA, timing uses `torch.cuda.Event` with explicit synchronisation to obtain accurate GPU-side measurements rather than host-side wall-clock time.

In [ ]:
@torch.no_grad()
def extract_features(model, dataloader):
    model.eval()
    all_features, all_pids, all_fnames = [], [], []
    for batch in tqdm(dataloader, desc='Extracting'):
        if len(batch) == 4:
            images, _labels, pids, fnames = batch
        else:
            images, anns = batch
            if isinstance(anns, dict):
                pids   = anns['pids'][:, 0].cpu()
                fnames = anns['img_name']
            else:
                pids   = torch.tensor([a['pids'][0] for a in anns])
                fnames = [a['img_name'] for a in anns]
        images = images.to(device)
        out    = model(images)
        feats  = out[0] if isinstance(out, tuple) else out
        feats  = F.normalize(feats, p=2, dim=1)
        all_features.append(feats.cpu())
        all_pids.append(pids.cpu() if isinstance(pids, torch.Tensor) else torch.tensor(pids))
        all_fnames.extend(fnames if isinstance(fnames, (list,tuple)) else fnames.tolist())
    return torch.cat(all_features, 0), torch.cat(all_pids, 0).numpy(), all_fnames


def evaluate_reid(distmat, q_pids, g_pids, q_fnames, g_fnames, exclude_self=True):
    all_cmc, all_AP, nv = [], [], 0
    for qi in range(distmat.shape[0]):
        qpid, qfname = q_pids[qi], q_fnames[qi]
        if not np.any(g_pids == qpid): continue
        order = np.argsort(distmat[qi])
        if exclude_self:
            order = np.array([i for i in order
                              if g_fnames[i] != qfname and
                              qfname.split('_',1)[-1] != g_fnames[i]])
        m = (g_pids[order] == qpid).astype(np.int32)
        if m.sum() == 0: continue
        nv += 1
        cmc = m.cumsum(); cmc[cmc>1] = 1; all_cmc.append(cmc[:50])
        tmp = m.cumsum()/(np.arange(len(m))+1.)*m
        all_AP.append(tmp.sum()/m.sum())
    if nv == 0:
        return {'mAP':0., 'rank1':0., 'rank5':0., 'rank10':0., 'num_valid_queries':0}
    cmc = np.array(all_cmc, dtype=np.float32).sum(0)/nv
    return {
        'mAP':   float(np.mean(all_AP)),
        'rank1': float(cmc[0]),
        'rank5': float(cmc[4]  if len(cmc)>4  else cmc[-1]),
        'rank10':float(cmc[9]  if len(cmc)>9  else cmc[-1]),
        'num_valid_queries': nv,
    }


def run_evaluation(model, q_loader, g_loader):
    print('\n' + '='*60 + '\nEVALUATION\n' + '='*60)
    qf, qp, qfn = extract_features(model, q_loader)
    gf, gp, gfn = extract_features(model, g_loader)
    distmat = (1 - torch.mm(qf, gf.t())).numpy()
    res = evaluate_reid(distmat, qp, gp, qfn, gfn,
                        exclude_self=cfg.exclude_query_from_gallery)
    print(f'  mAP    : {res["mAP"]*100:.2f}%')
    print(f'  Rank-1 : {res["rank1"]*100:.2f}%')
    print(f'  Rank-5 : {res["rank5"]*100:.2f}%')
    print(f'  Rank-10: {res["rank10"]*100:.2f}%')
    print(f'  Valid Q: {res["num_valid_queries"]}')
    print('='*60)
    return res


def profile_model(model, n_warmup=10, n_runs=100):
    model.eval()
    dummy = torch.randn(1, 3, cfg.img_height, cfg.img_width).to(device)
    macs, params = thop_profile(model, inputs=(dummy,), verbose=False)
    if device.type == 'cuda':
        s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(n_warmup): model(dummy)
        times = []
        with torch.no_grad():
            for _ in range(n_runs):
                s.record(); model(dummy); e.record()
                torch.cuda.synchronize()
                times.append(s.elapsed_time(e))
    else:
        with torch.no_grad():
            for _ in range(n_warmup): model(dummy)
        times = []
        with torch.no_grad():
            for _ in range(n_runs):
                t0 = time.perf_counter(); model(dummy)
                times.append((time.perf_counter()-t0)*1000)
    ms = float(np.mean(times))
    _, pfmt = clever_format([macs, params], '%.3f')
    print(f'  Params : {pfmt} ({int(params):,})')
    print(f'  GFLOPs : {macs*2/1e9:.2f}')
    print(f'  Latency: {ms:.2f} ms/img   {1000/ms:.1f} img/s')
    return {'total_params': int(params), 'flops_giga': macs*2/1e9,
            'inference_ms': ms, 'throughput': 1000./ms}

### Plotting Utilities

This section defines the visualisation functions used consistently across all experimental phases. A shared `PALETTE` of ten perceptually distinct colours ensures visual consistency when comparing multiple runs in the same figure. All figures are saved to `cfg.plots_dir` at 150 dpi when a `save_path` or `save_dir` is provided.

- `plot_training_history()`: produces a 2x3 grid of training diagnostics for a single fine-tuning run. The six panels cover epoch-level loss, batch-level loss with a moving-average smoother of window size `w = min(50, n_batches // 4)` to reduce noise, epoch-level and batch-level learning rate on a logarithmic scale (with vertical dashed lines marking epoch boundaries in the batch-level panel), and validation mAP and Rank-1 at the epochs where evaluation was performed. The logarithmic LR scale makes the warm-up and cosine decay phases immediately readable.

- `_radar_chart()`: renders a polar chart with four vertices (mAP, Rank-1, Rank-5, Rank-10), each scaled to the range [0, 100]. The polygon is closed by appending the first angle to the end of the angle list. Each model is drawn as a filled overlay with low alpha, allowing multiple models to be superimposed without occlusion. This view is particularly effective for identifying trade-offs between top-1 precision and recall at higher ranks.

- `plot_comparison()`: the main multi-run comparison function, called at the end of each phase. It produces three complementary views of the same result set:
  - a **grouped bar chart** with annotated bar labels comparing mAP and Rank-1 across all runs;
  - a **heatmap** of mAP indexed by fine-tuning strategy (rows) and loss function (columns), rendered only when the result dictionary contains keys in the expected `{strategy}_{loss}` format. Cells with no corresponding run are left blank (`NaN`). This view is most informative after Phase 2, when the full strategy-loss grid is populated;
  - a **radar chart** via `_radar_chart()` for a holistic four-metric comparison.
- **`plot_best_small_vs_base`** is the scale-up comparison function called after Phase 3.
  It automatically selects the best-performing ViT-Small run (highest mAP among
  `full_*`, `partial_*`, `freeze_*` keys) and the best ViT-Base run (`vitbase_*` keys),
  then produces a two-panel figure:
  - a grouped bar chart comparing seven dimensions side-by-side (mAP, Rank-1, Rank-5,
    Rank-10, latency in ms, GFLOPs, and parameter count in millions), with annotated bar
    labels for each value;
  - a delta table showing, for every metric, the Small value, the Base value, the signed
    delta, and a *Winner* column highlighted in blue (Small) or orange (Base) to make
    trade-offs immediately legible.
  The figure is saved as `best_small_vs_base.png` and constitutes the primary artefact for
  the cost-benefit analysis that motivates the small-first experimental design.



In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
rcParams.update({'font.size':10, 'axes.titlesize':12, 'figure.titlesize':13})
PALETTE = ['#2E86AB','#A23B72','#F18F01','#C73E1D','#6A994E',
           '#BC4B51','#5F4B8B','#E08D79','#1B998B','#E84855']


def plot_training_history(history, n_batches_per_epoch, num_epochs,
                          save_path=None, title_prefix=''):
    if not history['train_loss']:
        print('Empty history - nothing to plot.'); return

    epochs = range(1, len(history['train_loss'])+1)
    steps  = range(1, len(history['batch_loss'])+1)
    w = min(50, max(1, len(history['batch_loss'])//4))

    fig = plt.figure(figsize=(18, 10))
    fig.suptitle(f'{title_prefix}Training Curves', fontweight='bold')

    ax = plt.subplot(2,3,1)
    ax.plot(epochs, history['train_loss'], 'b-o', lw=2, ms=5)
    ax.set(xlabel='Epoch', ylabel='Loss', title='Loss / Epoch'); ax.grid(True, alpha=.3)

    ax = plt.subplot(2,3,2)
    ax.plot(steps, history['batch_loss'], 'b-', lw=.4, alpha=.3, label='Raw')
    if len(history['batch_loss']) >= w:
        sm = np.convolve(history['batch_loss'], np.ones(w)/w, mode='valid')
        ax.plot(range(w, len(history['batch_loss'])+1), sm, 'r-', lw=2, label=f'Smooth ({w})')
    ax.set(xlabel='Batch', ylabel='Loss', title='Loss / Batch')
    ax.grid(True, alpha=.3); ax.legend()

    ax = plt.subplot(2,3,3)
    ax.plot(epochs, history['lr'], 'r-o', lw=2, ms=5)
    ax.set(xlabel='Epoch', ylabel='LR', title='LR / Epoch')
    ax.set_yscale('log'); ax.grid(True, alpha=.3, which='both')

    ax = plt.subplot(2,3,4)
    ax.plot(steps, history['batch_lr'], 'r-', lw=1)
    ax.set(xlabel='Batch', ylabel='LR', title='LR / Batch')
    ax.set_yscale('log'); ax.grid(True, alpha=.3, which='both')
    for e in range(1, num_epochs):
        ax.axvline(e*n_batches_per_epoch, color='gray', ls='--', alpha=.25, lw=.8)

    ax = plt.subplot(2,3,5)
    if history['eval_map']:
        ax.plot(history['eval_epochs'], [m*100 for m in history['eval_map']],
                'b-o', lw=2, ms=8, label='mAP')
    ax.set(xlabel='Epoch', ylabel='mAP (%)', title='Val mAP')
    ax.grid(True, alpha=.3); ax.legend()

    ax = plt.subplot(2,3,6)
    if history['eval_rank1']:
        ax.plot(history['eval_epochs'], [r*100 for r in history['eval_rank1']],
                'g-s', lw=2, ms=8, label='Rank-1')
    ax.set(xlabel='Epoch', ylabel='Rank-1 (%)', title='Val Rank-1')
    ax.grid(True, alpha=.3); ax.legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved -> {save_path}')
    plt.show()


def _radar_chart(results_dict, title, save_path=None):
    metrics    = ['mAP', 'Rank-1', 'Rank-5', 'Rank-10']
    metric_keys= ['mAP',  'rank1',  'rank5',  'rank10']
    N          = len(metrics)
    angles     = [n/float(N)*2*np.pi for n in range(N)]
    angles    += angles[:1]                   # close the polygon

    fig, ax = plt.subplots(figsize=(7,7), subplot_kw=dict(polar=True))
    ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 100); ax.set_yticks([20,40,60,80,100])
    ax.set_yticklabels(['20','40','60','80','100'], fontsize=7, color='grey')
    ax.yaxis.set_tick_params(labelleft=True)

    names  = list(results_dict.keys())
    colors = PALETTE[:len(names)]
    for name, color in zip(names, colors):
        vals = [results_dict[name].get(mk, 0)*100 for mk in metric_keys]
        vals += vals[:1]
        ax.plot(angles, vals, '-o', lw=2, color=color, ms=6, label=name)
        ax.fill(angles, vals, alpha=0.08, color=color)

    ax.set_title(title, fontsize=13, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35,1.15), fontsize=9)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved -> {save_path}')
    plt.show()


def plot_comparison(results_dict, title='Comparison', save_dir=None):
    names  = list(results_dict.keys())
    map_v  = np.array([results_dict[n]['mAP']  *100 for n in names])
    r1_v   = np.array([results_dict[n]['rank1'] *100 for n in names])
    colors = PALETTE[:len(names)]

    # 1. Grouped bar chart
    fig, ax = plt.subplots(figsize=(max(10, len(names)*1.6), 6))
    x, w = np.arange(len(names)), 0.35
    b1 = ax.bar(x-w/2, map_v, w, label='mAP',    color='#2E86AB', alpha=.85, edgecolor='k', lw=.5)
    b2 = ax.bar(x+w/2, r1_v,  w, label='Rank-1', color='#F18F01', alpha=.85, edgecolor='k', lw=.5)
    for b in list(b1)+list(b2):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+.3,
                f'{b.get_height():.1f}', ha='center', va='bottom', fontsize=7.5)
    ax.set(xticks=x, xticklabels=names, ylabel='Score (%)', title=f'{title} - mAP & Rank-1')
    plt.xticks(rotation=30, ha='right'); ax.legend(); plt.tight_layout()
    if save_dir:
        plt.savefig(os.path.join(save_dir, f'{title.replace(" ","_")}_bar.png'),
                    dpi=150, bbox_inches='tight')
    plt.show()

    # 2. mAP heatmap (strategy x loss)
    strategies = ['full', 'partial', 'freeze']
    loss_names = ['triplet', 'arcface', 'ntxent']
    hmap = np.full((len(strategies), len(loss_names)), np.nan)
    for i, s in enumerate(strategies):
        for j, l in enumerate(loss_names):
            k = f'{s}_{l}'
            if k in results_dict: hmap[i,j] = results_dict[k]['mAP']*100
    if not np.all(np.isnan(hmap)):
        fig, ax = plt.subplots(figsize=(7,4))
        im = ax.imshow(hmap, cmap='YlOrRd', aspect='auto')
        ax.set_xticks(range(len(loss_names)));  ax.set_xticklabels(loss_names, fontweight='bold')
        ax.set_yticks(range(len(strategies)));  ax.set_yticklabels(strategies, fontweight='bold')
        for i in range(len(strategies)):
            for j in range(len(loss_names)):
                if not np.isnan(hmap[i,j]):
                    ax.text(j, i, f'{hmap[i,j]:.1f}%', ha='center', va='center',
                            fontsize=11, fontweight='bold',
                            color='black' if hmap[i,j]>60 else 'black')
        plt.colorbar(im, ax=ax, label='mAP (%)')
        ax.set_title('mAP: Strategy × Loss', fontweight='bold')
        plt.tight_layout()
        if save_dir:
            plt.savefig(os.path.join(save_dir, f'{title.replace(" ","_")}_heatmap.png'),
                        dpi=150, bbox_inches='tight')
        plt.show()

    # 3. Radar chart (4 metrics)
    radar_save = os.path.join(save_dir, f'{title.replace(" ","_")}_radar.png') if save_dir else None
    _radar_chart(results_dict, title=f'{title} - Radar', save_path=radar_save)

def plot_best_small_vs_base(results_dict, save_dir=None):
    """
    Compare best ViT-Small (full_*, partial_*, freeze_*) vs best ViT-Base (vit_base_*) by mAP.
    """
    import matplotlib.patheffects as pe

    base_keys  = {k: v for k, v in results_dict.items() if k.startswith('vit_base_')}
    small_keys = {k: v for k, v in results_dict.items()
                  if any(k.startswith(s) for s in ('full_', 'partial_', 'freeze_'))
                  and not k.startswith('vit_base_')}

    if not small_keys or not base_keys:
        print('[plot_best_small_vs_base] Not enough models to compare.'); return

    best_small_k = max(small_keys, key=lambda k: small_keys[k]['mAP'])
    best_base_k  = max(base_keys,  key=lambda k: base_keys[k]['mAP'])
    best_small   = results_dict[best_small_k]
    best_base    = results_dict[best_base_k]

    metrics = {
        'mAP (%)':      ('mAP',          100),
        'Rank-1 (%)':   ('rank1',         100),
        'Rank-5 (%)':   ('rank5',         100),
        'Rank-10 (%)':  ('rank10',        100),
        'Latency (ms)': ('inference_ms',    1),
        'GFLOPs':       ('flops_giga',      1),
        'Params (M)':   ('total_params',  1e-6),
    }

    labels = list(metrics.keys())
    vals_s = [best_small.get(v, 0) * m for v, m in metrics.values()]
    vals_b = [best_base.get(v,  0) * m for v, m in metrics.values()]
    x, w   = np.arange(len(labels)), 0.35

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle('Best Small vs. Best Base Architecture', fontweight='bold')

    #Grouped bar chart#
    ax = plt.subplot(2, 1, 1)
    b_s = ax.bar(x - w/2, vals_s, w, label=f'ViT-Small  ({best_small_k})',
                 color='#2E86AB', alpha=.85, edgecolor='k', lw=.5)
    b_b = ax.bar(x + w/2, vals_b, w, label=f'ViT-Base   ({best_base_k})',
                 color='#F18F01', alpha=.85, edgecolor='k', lw=.5)

    for b in list(b_s) + list(b_b):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + .3,
                f'{b.get_height():.1f}', ha='center', va='bottom', fontsize=7.5,
                path_effects=[pe.withStroke(linewidth=1.8, foreground='white')])

    ax.set(xticks=x, xticklabels=labels, ylabel='Score / Value',
           title='Metric Comparison - mAP, Rank-k, Efficiency')
    plt.xticks(rotation=20, ha='right')
    ax.grid(True, alpha=.3)
    ax.legend()

    #Delta table
    ax2 = plt.subplot(2, 1, 2)
    ax2.axis('off')

    higher_better = {'mAP (%)', 'Rank-1 (%)', 'Rank-5 (%)', 'Rank-10 (%)'}
    col_labels = ['Metric', f'Small ({best_small_k})', f'Base ({best_base_k})', 'Delta', 'Best']
    table_data = []
    for lbl, vs, vb in zip(labels, vals_s, vals_b):
        delta  = vb - vs
        sign   = f'+{delta:.2f}' if delta >= 0 else f'{delta:.2f}'
        if lbl in higher_better:
            winner = 'Base' if delta > 0 else ('Small' if delta < 0 else '=')
        else:
            # lower is better
            winner = 'Small' if delta > 0 else ('Base' if delta < 0 else '=')
        table_data.append([lbl, f'{vs:.2f}', f'{vb:.2f}', sign, winner])

    tbl = ax2.table(
        cellText=table_data,
        colLabels=col_labels,
        cellLoc='center',
        loc='center',
        bbox=[0.05, 0, 0.90, 1]
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)

    for j in range(len(col_labels)):
        tbl[0, j].set_facecolor('#2E86AB')
        tbl[0, j].set_text_props(color='white', fontweight='bold')

    for i, row in enumerate(table_data, start=1):
        winner_val = row[-1]
        bg = '#f0f4f8' if i % 2 == 0 else 'white'
        for j in range(len(col_labels)):
            cell = tbl[i, j]
            cell.set_facecolor(bg)
            cell.set_edgecolor('#cccccc')
        if winner_val == 'Base':
            tbl[i, 4].set_facecolor('#fde8d0')
        elif winner_val == 'Small':
            tbl[i, 4].set_facecolor('#d6e8f7')

    plt.tight_layout()
    if save_dir:
        out = os.path.join(save_dir, 'best_small_vs_base.png')
        plt.savefig(out, dpi=150, bbox_inches='tight')
        print(f'Saved -> {out}')
    plt.show()

## 5. Ablation Phases

### Phase 0 - Pretrained Models Evaluation

Before any fine-tuning, all four **ViT-Small** checkpoints are evaluated in a zero-shot
regime on PRW. No parameter update is performed: the models are loaded with their
source-domain weights and evaluated directly on the target domain. This phase serves two
purposes. First, it establishes a lower-bound baseline that quantifies how much
discriminative information each source domain transfers to PRW without adaptation. Second,
it identifies the best-performing ViT-Small initialisation, which is used as the starting
point for all subsequent ablation phases (Phases 1 and 2) and, indirectly, as the reference
domain for selecting the ViT-Base checkpoint in Phase 3: because both architectures share
the same pre-training datasets, the source domain that maximises zero-shot mAP on ViT-Small
is a reliable proxy for the best initialisation on ViT-Base as well.

The four source domains under evaluation are:

| Key          | Dataset            | Notes                                        |
|--------------|--------------------|----------------------------------------------|
| `duke`       | DukeMTMC-ReID      | Multi-camera outdoor campus, 8 cameras       |
| `market1501` | Market-1501        | Multi-camera outdoor campus, 6 cameras       |
| `msmt`       | MSMT17             | Large-scale multi-environment, 15 cameras    |
| `occduke`    | Occluded-DukeMTMC  | Extension of Duke with heavy occlusion       |

Each model is instantiated via `create_personvit_model` with `cfg.vit_small_transformer_type`
and `cfg.vit_small_embedding_dim`, evaluated with `run_evaluation`, and profiled with
`profile_model`. Results are persisted to disk immediately after each model to avoid data
loss in case of session interruption. The model is explicitly deleted and
`torch.cuda.empty_cache()` is called after each evaluation to free VRAM before loading the
next checkpoint. At the end of the loop, the checkpoint with the highest mAP is selected as
`best_pretrain_key` and used as the initialisation for all ViT-Small fine-tuning runs in
Phases 1 and 2.



In [ ]:
print('\n' + '='*70 + f'\nEVALUATING {len(cfg.pretrained_models_small)} PRETRAINED MODELS\n' + '='*70)

for model_key, model_info in cfg.pretrained_models_small.items():
    print(f'\n-- {model_info["display_name"]} ({model_key})')
    if not os.path.exists(model_info['path']):
        print(f'  WARNING: weights not found, skipping.'); continue
    try:
        model  = create_personvit_model(model_info['path'], cfg.vit_small_transformer_type,
                                        cfg.vit_small_embedding_dim, train_reid.num_classes, reid_cfg)
        res    = run_evaluation(model, query_loader, test_loader)
        hw     = profile_model(model)
        RESULTS[model_key] = {**res, **hw, 'display_name': model_info['display_name'],
                              'strategy': 'Pretrained', 'loss': '--'}
        save_results()                              # persist immediately
        del model; torch.cuda.empty_cache()
    except Exception:
        import traceback; traceback.print_exc()

best_pre = max((k for k in cfg.pretrained_models_small if k in RESULTS), key=lambda k: RESULTS[k]['mAP'])
print(f'\nBest pretrained: {best_pre}  (mAP {RESULTS[best_pre]["mAP"]*100:.2f}%)')
print('This checkpoint will be used for fine-tuning.')

In [ ]:
pretrained_results = {k: RESULTS[k] for k in cfg.pretrained_models_small if k in RESULTS}
plot_comparison(pretrained_results, title='Pretrained Models', save_dir=str(cfg.plots_dir))

### Phase 1 - Fine-Tuning Strategy Comparison

Phase 1 investigates how the choice of fine-tuning strategy affects Re-ID performance
when adapting the best pretrained **ViT-Small** checkpoint to PRW. Running the strategy
ablation on the smaller architecture (22 M parameters) rather than ViT-Base (86 M) keeps
each training run approximately 4x faster, making it practical to compare all three
strategies within a single Kaggle session. The metric learning loss is fixed to ArcFace
throughout this phase so that any difference in results can be attributed exclusively to
the strategy and not to the loss function. Three strategies are compared:

| Strategy       | Backbone   | Head            | LR   | Notes                                                                 |
|----------------|------------|-----------------|------|-----------------------------------------------------------------------|
| Full FT        | Trainable  | Trainable       | 1e-5 | All parameters updated; low LR to avoid catastrophic forgetting       |
| Partial FT     | Frozen     | Trainable       | 3e-4 | Backbone weights preserved; only BNNeck and projection layer updated  |
| FreezeRetrain  | Frozen     | Re-initialised  | 3e-4 | Head weights reset before training; tests how much discriminative power the frozen backbone alone provides |

`STRATEGY_CFG` is a configuration dictionary that maps each strategy key to its complete
set of hyperparameters and the corresponding freeze function. This design allows
`run_finetuning` to apply any strategy by simply looking up its entry, without any
conditional branching in the training code itself.

The three runs are executed sequentially via `run_finetuning` on ViT-Small. At the end
of the phase, the strategy with the highest mAP is selected as `best_strategy` and
carried forward to Phase 2, where it is held fixed while the loss function is varied.


In [ ]:
STRATEGY_CFG = {
    'full': {
        'freeze_fn':    unfreeze_all,
        'reset_head':   False,
        'lr':           cfg.full_lr,    'wd': cfg.full_wd,
        'epochs':       cfg.full_epochs,
        'warmup_ep':    cfg.full_warmup_ep,
        'warmup_factor':cfg.full_warmup_factor,
        'label':        'Full FT',
    },
    'partial': {
        'freeze_fn':    freeze_backbone_unfreeze_head,
        'reset_head':   False,
        'lr':           cfg.partial_lr, 'wd': cfg.partial_wd,
        'epochs':       cfg.partial_epochs,
        'warmup_ep':    cfg.partial_warmup_ep,
        'warmup_factor':cfg.partial_warmup_factor,
        'label':        'Partial FT',
    },
    'freeze': {
        'freeze_fn':    freeze_all_backbone,
        'reset_head':   True,
        'lr':           cfg.freeze_lr,  'wd': cfg.freeze_wd,
        'epochs':       cfg.freeze_epochs,
        'warmup_ep':    cfg.freeze_warmup_ep,
        'warmup_factor':cfg.freeze_warmup_factor,
        'label':        'Freeze+Retrain',
    },
}

In [ ]:
for strategy in ['full', 'partial', 'freeze']:
    run_finetuning(strategy, 'arcface')

In [ ]:
# Phase 1 comparison plots
phase1_results = {k: RESULTS[k] for k in ['full_arcface','partial_arcface','freeze_arcface']
                  if k in RESULTS}
best_strategy  = max(phase1_results, key=lambda k: phase1_results[k]['mAP']).split('_')[0]
print(f'Phase 1 winner: strategy="{best_strategy}"  '
      f'(mAP {RESULTS[best_strategy+"_arcface"]["mAP"]*100:.2f}%)')

plot_comparison(phase1_results, title='Phase 1 - Strategy Comparison',
                save_dir=str(cfg.plots_dir))

### Phase 2 - Metric Learning Loss Comparison

Phase 2 investigates the effect of the metric learning loss function on Re-ID performance.
The fine-tuning strategy is fixed to the winner of Phase 1 (`best_strategy`), so that any
difference in results is attributable exclusively to the loss. Three losses are compared:

| Loss Type         | Family       | Miner         | Key property |
|-------------------|--------------|---------------|--------------|
| `TripletMarginLoss` | Pair-based | `BatchHardMiner` | Optimises relative distances between anchor, positive, and hardest negative in the batch |
| `ArcFaceLoss`     | Proxy-based  | None          | Maintains a learnable class weight matrix; maximises angular margin between identity embeddings in a closed-set formulation |
| `NTXentLoss`      | Contrastive  | None (implicit) | Maximises cosine similarity between positive pairs via a softmax over all in-batch negatives, scaled by temperature `cfg.ntxent_temperature`; derived from SimCLR |

Note that the `arcface` run is not repeated here: it was already executed in Phase 1 and
its result is retrieved directly from `RESULTS`. Only `triplet` and `ntxent` are new runs,
keeping the total number of training runs minimal.

At the end of the phase, the loss with the highest mAP is selected as `best_loss` and
carried forward to Phase 3, where the winning configuration (strategy + loss) is scaled up
to ViT-Base for the multi-scale comparison.



In [ ]:
for loss_type in ['triplet', 'ntxent']:
    run_finetuning(best_strategy, loss_type)

In [ ]:
# Phase 1 comparison plots
phase2_keys    = [f'{best_strategy}_{l}' for l in ['triplet','arcface','ntxent']]
phase2_results = {k: RESULTS[k] for k in phase2_keys if k in RESULTS}
best_loss      = max(phase2_results, key=lambda k: phase2_results[k]['mAP']).split('_',1)[1]
print(f'Phase 2 winner: loss="{best_loss}"  '
      f'(mAP {RESULTS[best_strategy+"_"+best_loss]["mAP"]*100:.2f}%)')

plot_comparison(phase2_results, title='Phase 2 - Loss Comparison',
                save_dir=str(cfg.plots_dir))

### Phase 3 - ViT-Small Fine-Tuning

Phase 3 scales up the winning configuration identified in Phases 1 and 2 to
**ViT-Base** (embedding dimension 768, 12 attention heads, 86 M parameters), using the corresponding Market-1501 pretrained checkpoint of the larger architecture. The fine-tuning strategy (`best_strategy`) and loss function (`best_loss`) are held fixed at the values selected during the ViT-Small ablation, so the only variable introduced in this phase is the model scale. The run is registered under the key `vitbase_{best_strategy}_{best_loss}` via `run_key_override` to distinguish it from the ViT-Small runs in `RESULTS` while preserving the same naming convention.

This *scale-up last* design is the direct consequence of the small-first experimental choice described in the Introduction: running the full ablation grid on the cheaper ViT-Small first reduces total GPU consumption significantly, and only once the optimal configuration is known does the study commit to a single, definitive ViT-Base training run. The primary objective of Phase 3 is therefore not to repeat the ablation at larger scale, but to quantify whether the mAP gain from scaling up justifies the increase in computational cost. The comparison covers four dimensions: retrieval performance (mAP, Rank-1, Rank-5, Rank-10), parameter count, computational cost (GFLOPs), and inference latency, all of which are aggregated and visualised by `plot_best_small_vs_base` in the Final Comparison section.



In [ ]:
run_finetuning(
    strategy_key      = best_strategy,
    loss_type         = best_loss,
    transformer_type  = cfg.vit_base_transformer_type,
    embedding_dim     = cfg.vit_base_embedding_dim,
    pretrain_path     = cfg.pretrained_models[cfg.best_pretrain_key]['path'],
    run_key_override  = f'vit_base_{best_strategy}_{best_loss}',
)

## 6. Final Comparison

This section aggregates the results of all runs executed throughout the notebook into a single unified view, covering pretrained baselines (Phase 0), fine-tuning strategy runs (Phase 1), loss comparison runs (Phase 2), and the ViT-Base scale-up run (Phase 3).

The aggregation logic selects keys from `RESULTS` in two groups:

- **Pretrained baselines**: all keys matching `cfg.pretrained_models_small`
  (i.e. `duke`, `market1501`, `msmt`, `occduke`), which represent zero-shot ViT-Small   performance on PRW without any fine-tuning.
- **Fine-tuned runs**: all keys matching the prefixes `full_*`, `partial_*`, `freeze_*`, and `vitbase_*`, covering both the ViT-Small ablation runs and the final ViT-Base scale-up.

`plot_comparison` is called on the full combined dictionary to produce a grouped bar chart, a strategy-vs-loss heatmap, and a radar chart spanning all models simultaneously. This global view makes it immediately apparent how much fine-tuning improves over the pretrained baselines and how the ViT-Base run positions itself relative to the ViT-Small ablation grid.

The full results table is then serialised to `allresults.csv` in `cfg.results_dir`. Each row corresponds to one run and includes the following columns: `mAP`, `Rank-1`, `Rank-5`, `Rank-10`, `Total Params (M)`, `Trainable Params`, `GFLOPs`, `Latency (ms)`, and `Throughput (img/s)`. The table is sorted by mAP in descending order to make the best-performing configuration immediately visible at the top.

In [ ]:
ft_keys   = [k for k in RESULTS if any(k.startswith(s) for s in ('full_','partial_','freeze_','vit_base_'))]
all_keys  = list(cfg.pretrained_models.keys()) + ft_keys
all_res   = {k: RESULTS[k] for k in all_keys if k in RESULTS}

plot_comparison(all_res, title='All Models', save_dir=str(cfg.plots_dir))

rows = []
for rk, v in RESULTS.items():
    rows.append({
        'Run':                rk,
        'Strategy':           v.get('strategy',    '?'),
        'Loss':               v.get('loss',         '--'),
        'mAP (%)':            round(v['mAP']  *100, 2),
        'Rank-1 (%)':         round(v['rank1']*100, 2),
        'Rank-5 (%)':         round(v['rank5']*100, 2),
        'Rank-10 (%)':        round(v['rank10']*100, 2),
        'Total Params (M)':   round(v.get('total_params', 0)/1e6, 1),
        'Trainable Params':   v.get('trainable_params', 'N/A'),  # only for FT runs
        'GFLOPs':             round(v.get('flops_giga', 0), 2),
        'Latency (ms)':       round(v.get('inference_ms', 0), 2),
        'Throughput (i/s)':   round(v.get('throughput', 0), 1),
    })
df = (pd.DataFrame(rows)
        .set_index('Run')
        .sort_values('mAP (%)', ascending=False))
print('\n' + '='*110)
print('COMPLETE RESULTS (sorted by mAP)')
print('='*110)
print(df.to_string())
df.to_csv(cfg.results_dir / 'all_results.csv')
print(f'\nCSV saved to {cfg.results_dir / "all_results.csv"}')

### ViT-Small vs ViT-Base: Scale-Up Analysis

This cell calls `plot_best_small_vs_base(RESULTS)`, the dedicated comparison function that materialises the core result of the small-first experimental design: a direct, quantitative confrontation between the best ViT-Small run and the ViT-Base scale-up.

The function automatically identifies the two candidates from `RESULTS`:
- **Best ViT-Small**: the run with the highest mAP among all keys with prefix `full_*`, `partial_*`, or `freeze_*`.
- **Best ViT-Base**: the run with the highest mAP among all keys with prefix `vitbase_*`.

The output is a two-panel figure saved as `best_small_vs_base.png`:

1. **Grouped bar chart**: compares seven dimensions side by side for both models:  `mAP`, `Rank-1`, `Rank-5`, `Rank-10` (higher is better), `Latency (ms)`, `GFLOPs`, and `Params (M)` (lower is better). Each bar is annotated with its numeric value.

2. **Delta table**: for every metric reports the ViT-Small value, the ViT-Base value, the signed delta (Base minus Small), and a *Best* column colour-coded in blue for Small and orange for Base, making the winner immediately readable at a glance. For accuracy metrics the winner is the model with the higher value; for efficiency metrics (latency, GFLOPs, parameters) the winner is the model with the lower value.


In [ ]:
plot_best_small_vs_base(RESULTS)

## 7. Conclusions

### Summary of Results

The ablation study yields ten configurations spanning zero-shot evaluation of four source domains, three fine-tuning strategies, three loss functions on ViT-Small, and a scale-up run on ViT-Base. The results are consistent with established findings in metric learning for person re-identification, and each phase produces outcomes that are largely expected from a methodological standpoint, with a few noteworthy observations regarding the relative contribution of loss function choice and the cost-benefit profile of the scale-up.

### Effect of Source Domain (Phase 0)

The four pretrained ViT-Small baselines evaluated in zero-shot on PRW span a range of 24.7 pp in mAP (51.58% for Duke to 75.26% for Market-1501), revealing a strong sensitivity to source-domain choice even before any fine-tuning. The ranking Market-1501 > MSMT17 > Occluded-Duke > Duke is consistent with the degree of visual and environmental overlap with PRW: Market-1501 and PRW are both multi-camera outdoor campus datasets with similar camera heights, pedestrian densities, and lighting conditions, which explains why the Market-1501 checkpoint transfers substantially better than the others. The Duke and Occluded-Duke checkpoints perform significantly worse (51.58% and 53.29%), likely because the DukeMTMC environment differs in camera placement and average occlusion level from the PRW test distribution. MSMT17 (55.51%) occupies an intermediate position: its greater domain diversity provides more general features than Duke but does not match the direct domain proximity of Market-1501.

The gap between the best and worst pretrained checkpoints (+23.7 pp mAP) is considerably larger than the gap introduced by any subsequent fine-tuning decision, which underscores the dominant role of source-domain selection in transfer learning for Re-ID. Market-1501 is selected as `best_pretrain_key` and used as the initialisation for all fine-tuning runs in Phases 1 and 2.

### Effect of Fine-Tuning Strategy (Phase 1)

Phase 1 compares three fine-tuning strategies applied to the Market-1501 ViT-Small checkpoint with ArcFace loss fixed. Full FT (`full_arcface`, mAP = 78.10%, Rank-1 = 93.39%) outperforms both Partial FT (`partial_arcface`, mAP = 75.62%, Rank-1 = 93.00%) and Freeze+Retrain (`freeze_arcface`, mAP = 75.64%, Rank-1 = 93.19%), with differences of +2.48 pp and +2.46 pp in mAP respectively. The Rank-1 differences are smaller (+0.39 pp and +0.20 pp), suggesting that the three strategies converge on similar top-1 retrieval accuracy but diverge more substantially on the quality of the full ranked list, which mAP is more sensitive to.

A notable observation is that Freeze+Retrain slightly outperforms Partial FT in mAP (75.64% vs. 75.62%) despite having 384 trainable parameters compared to 768. The margin is well within expected noise for a 20-epoch run, and no causal interpretation should be drawn from it. What is meaningful is that both head-only strategies plateau near the zero-shot Market-1501 baseline (75.26%), implying that the backbone features alone, without any end-to-end adaptation, are sufficient to approximately recover zero-shot performance but insufficient to improve substantially beyond it. Full FT, which updates all 22 M parameters at the lower learning rate of 1e-5 to mitigate catastrophic forgetting, is the only strategy that consistently exceeds the zero-shot baseline by a meaningful margin (+2.84 pp mAP). Full FT is selected as `best_strategy`.

### Effect of Loss Function (Phase 2)

Phase 2 compares three metric learning objectives under the Full FT strategy. TripletMarginLoss with BatchHardMiner achieves the best mAP (81.50%, Rank-1 = 93.44%), followed by NTXentLoss (80.77%, Rank-1 = 93.15%) and ArcFaceLoss (78.10%, Rank-1 = 93.39%). The ordering is consistent across all four retrieval metrics.

The +3.40 pp advantage of TripletMarginLoss over ArcFaceLoss is the largest gain observed within Phase 2 and deserves careful interpretation. ArcFaceLoss operates in a closed-set regime: it maintains a learnable weight matrix over the 483 training identities and maximises angular margins between classes. At test time, however, PRW Re-ID is an open-set retrieval problem where the query identities are entirely disjoint from the training set. This mismatch between the closed-set training objective and the open-set evaluation protocol penalises ArcFace: the learnable class weights introduce a source-domain bias that does not generalise to unseen identities. TripletMarginLoss, by contrast, directly optimises relative distances in the embedding space without reference to a fixed class vocabulary, making it structurally better suited to the open-set deployment regime. NTXentLoss occupies an intermediate position: its implicit in-batch negative sampling approximates the triplet objective but applies a uniform temperature-scaled softmax over all negatives regardless of their hardness, which is less efficient than the hard-negative mining of BatchHardMiner at the batch sizes used here (128 crops). TripletMarginLoss with BatchHardMiner is selected as `best_loss`.

### Scale-Up: ViT-Base (Phase 3)

Scaling from ViT-Small to ViT-Base with the winning configuration (Full FT, Triplet) produces a consistent improvement across all retrieval metrics: mAP rises from 81.50% to 85.65% (+4.15 pp), Rank-1 from 93.44% to 94.51% (+1.07 pp), Rank-5 from 97.76% to 98.49% (+0.73 pp), and Rank-10 from 98.30% to 99.12% (+0.82 pp). The mAP gain is the most practically significant: it implies that ViT-Base not only retrieves the correct identity at the top of the list more often than ViT-Small, but also ranks multiple correct gallery matches higher across the full retrieval list, which is the behaviour mAP is specifically designed to capture.

This improvement comes at a substantial computational cost. ViT-Base has 3.93x more parameters (86.5 M vs. 22.0 M), 3.97x higher GFLOPs (22.08 vs. 5.56), and 1.68x higher per-image latency (11.80 ms vs. 7.02 ms), corresponding to a 40.4% reduction in throughput (84.8 vs. 142.4 images/s). The latency ratio is considerably lower than the GFLOPs ratio because ViT-Base benefits disproportionately from T4 Tensor Core utilisation at larger matrix dimensions: the wider attention heads and higher embedding dimension allow better hardware occupancy, partially amortising the theoretical compute cost.

### Model Selection for Person Search

From the perspective of the downstream two-stage person search pipeline, the choice between ViT-Small and ViT-Base involves an explicit trade-off between Re-ID accuracy and inference throughput. ViT-Base is the preferred candidate when retrieval quality is the primary concern and the deployment hardware supports the additional compute: its mAP of 85.65% and Rank-1 of 94.51% represent the strongest result in this study. ViT-Small (Full FT, Triplet) is the viable alternative in throughput-constrained deployments, where its mAP of 81.50% and latency of 7.02 ms per crop represent a favourable operating point: the 4.15 pp mAP penalty relative to ViT-Base is offset by a 1.68x reduction in inference time, which is non-negligible when the Re-ID module must process all detected crops in real-time surveillance footage.

It is also worth noting that the best ViT-Small fine-tuned run (81.50% mAP) exceeds the best pretrained baseline by +6.24 pp, confirming that target-domain fine-tuning provides a consistent and reproducible benefit independent of model scale. The small-first ablation design adopted in this notebook is therefore validated: the configuration that maximised mAP on ViT-Small (Full FT, Triplet) transferred directly to ViT-Base without any additional hyperparameter search, producing the best result in the study with a single scale-up run.

### A Note on Data Leakage

The retrieval performance reported in this study, in particular the mAP of 85.65% achieved by ViT-Base and the strong zero-shot baseline of 75.26% for the Market-1501 pretrained checkpoint, may raise a legitimate concern about data leakage between the pre-training source domain (Market-1501) and the evaluation target domain (PRW). The suspicion is reasonable: both datasets were collected on the campus of Tsinghua University by the same research group using the same six cameras in the summer of 2014, and the identity labels in PRW are assigned by cross-referencing subjects with Market-1501 annotations.

However, both the official PRW project page and the paper (Zheng et al., CVPR 2017 and https://zheng-lab-anu.github.io/Project/project_prw.html) rule out any identity overlap explicitly. The project page states:

> *"The PRW dataset is annotated from the same video as the Market-1501 dataset [...] readers should keep in mind that the identities in PRW are not corresponded to those in Market-1501."*

The paper (Section 3.1) further clarifies the annotation procedure:

> *"We first manually draw a bounding box for all pedestrians who appear in the frames and then assign an ID if it exists in the Market-1501 dataset."*

Taken together, these two sources confirm that while PRW and Market-1501 share the same raw video footage and camera infrastructure, their identity sets are completely disjoint: no identity in PRW corresponds to any identity in Market-1501. The shared video source means that the same physical individuals may appear in both datasets, but they are assigned different and unrelated labels in each benchmark, making cross-dataset identity leakage structurally impossible.

The elevated zero-shot performance of the Market-1501 checkpoint is therefore attributable to strong domain alignment — shared environment, camera hardware, resolution, and scene statistics — rather than to any form of identity or image-level leakage. The consistent gain from fine-tuning (+6.24 pp mAP for ViT-Small, +10.39 pp for ViT-Base over the zero-shot baseline) further confirms that PRW represents a genuinely novel distribution from the model's perspective: a checkpoint that had memorised PRW identities during pre-training would not benefit this systematically from additional target-domain adaptation.


## References

[1] Hu, B., Wang, X., Liu, W. (2024). *PersonViT: Large-scale Self-supervised Vision Transformer for Person Re-Identification*. Machine Vision and Applications, 2025. DOI: [10.1007/s00138-025-01659-y](https://doi.org/10.1007/s00138-025-01659-y) — arXiv: [2408.05398](https://arxiv.org/abs/2408.05398)

[2 hustvl. *PersonViT — Official GitHub Repository*. [https://github.com/hustvl/PersonViT](https://github.com/hustvl/PersonViT)

[3] lakeAGI. *PersonViT Pre-trained Weights (ViT-Base and ViT-Small)*. Hugging Face Model Hub. [https://huggingface.co/lakeAGI/PersonViTReID/tree/main](https://huggingface.co/lakeAGI/PersonViTReID/tree/main)

[4] Zheng, L., Zhang, H., Sun, S., Chandraker, M., Tian, Q. (2017). *Person Re-identification in the Wild*. In Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition (CVPR), 2017. arXiv: [1604.02531](https://arxiv.org/abs/1604.02531)
